# DRLB vs Baselines on Standard Test Split

This notebook runs DRLB on `RND42N10Config`, which uses the same standard train/test split as the baseline experiment notebook.

It then compares the fresh DRLB result with the saved baseline results from `outputs/results_baselines.csv`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

notebook_dir = Path().resolve()
example_notebooks_dir = notebook_dir.parent.parent
bat_autobidding_dir = example_notebooks_dir.parent

if str(example_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(example_notebooks_dir))
if str(bat_autobidding_dir) not in sys.path:
    sys.path.insert(0, str(bat_autobidding_dir))

from experiments.exp_configs import RND42N10Config
from experiments.drlb_experiment import opt_search_drlb, train_best_drlb, evaluate_drlb


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = RND42N10Config()
config.ensure_artifact_dirs()

stats_df = pd.read_csv(config.data_config["train"]["stats_path"])
n_trials = 1 # Increase if you want a stronger DRLB tune.

print("Using train stats:", config.data_config["train"]["stats_path"])
print("Using test stats:", config.data_config["test"]["stats_path"])
print("Train rows:", len(stats_df))

Using train stats: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_filtered_train_final.csv
Using test stats: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_filtered_test_final.csv
Train rows: 1533171


In [3]:
study, best_params_path = opt_search_drlb(
    config=config,
    stats_df=stats_df,
    n_trials=n_trials,
    verbose=False,
)

best_model_path = config.best_models_dir / f"drlb_{config.metric.lower()}_{config.auction_mode}.pt"

_ = train_best_drlb(
    stats_df=stats_df,
    best_params_path=str(best_params_path),
    model_path=str(best_model_path),
    config=config,
    verbose=False,
)

res = evaluate_drlb(
    config=config,
    model_path=str(best_model_path),
    best_params_path=str(best_params_path),
    verbose=False,
)

print("Best trial value:", float(study.best_trial.value))
print("DRLB score:", res["score"])

[I 2026-03-29 22:43:48,897] A new study created in memory with name: no-name-11d5dcaf-7b81-4fce-af65-218c93a3c196
[I 2026-03-29 22:45:28,338] Trial 0 finished with value: 159.21241976902388 and parameters: {'max_bid': 43.284502212938804, 'T': 93, 'lambda_min': 2.910635913133059e-09, 'lambda_max': 7.6611007077713635}. Best is trial 0 with value: 159.21241976902388.


CPC_REL: 9634.24127635161, rmse: 1.236140792875836, clicks_sum: 159.21241976902388, quickspend: 0.0
Best trial:
  Value: 159.21241976902388
  Params:
    max_bid: 43.284502212938804
    T: 93
    lambda_min: 2.910635913133059e-09
    lambda_max: 7.6611007077713635


KeyboardInterrupt: 

In [ ]:
baseline_path = config.outputs_dir / "results_baselines.csv"

drlb_row = pd.DataFrame([
    {
        "model": "drlb",
        "SCR": res["score"][2],
        "CPC_REL": res["score"][0],
        "RMSE": res["score"][1],
        "quickspend": res["score"][3],
        "source": "fresh_drlb_run",
    }
])

if baseline_path.exists():
    baselines_df = pd.read_csv(baseline_path)
    baseline_comp = baselines_df[["model", "SCR"]].copy()
    baseline_comp["CPC_REL"] = pd.NA
    baseline_comp["RMSE"] = pd.NA
    baseline_comp["quickspend"] = pd.NA
    baseline_comp["source"] = "results_baselines.csv"
    comparison_df = pd.concat([baseline_comp, drlb_row], ignore_index=True, sort=False)
else:
    comparison_df = drlb_row.copy()

comparison_df = comparison_df.sort_values("SCR", ascending=False).reset_index(drop=True)
comparison_path = config.outputs_dir / "results_drlb_vs_baselines.csv"
comparison_df.to_csv(comparison_path, index=False)

comparison_df

/var/folders/ht/mcd64cts6p959c6g8xqp_jh00000gn/T/ipykernel_14848/1720978922.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  comparison_df = pd.concat([baseline_comp, drlb_row], ignore_index=True, sort=False)


,model,SCR,CPC_REL,RMSE,quickspend,source
0,linear,38000.766232,NaN,NaN,NaN,results_baselines.csv
1,tapid,33976.186420,NaN,NaN,NaN,results_baselines.csv
2,mpid,20718.559629,NaN,NaN,NaN,results_baselines.csv
3,broi,16290.103991,NaN,NaN,NaN,results_baselines.csv
4,drlb,159.212420,9634.241276,1.236141,0.0,fresh_drlb_run


In [ ]:
comparison_path = config.outputs_dir / "results_drlb_vs_baselines.csv"
report_path = config.outputs_dir / "drlb_fix_report.md"

print("Comparison CSV:", comparison_path)
print("DRLB report:", report_path)
comparison_path.exists(), report_path.exists()

Comparison CSV: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_1_rnd_42_n10/outputs/results_drlb_vs_baselines.csv
DRLB report: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_1_rnd_42_n10/outputs/drlb_fix_report.md


(True, True)